## 09_Time_Based_Feature_Store_Integration

Explainable AI Credit Risk Decision Platform Integrating Structured Borrower Data, NLP-Driven Text Intelligence, and Macroeconomic Indicators for Transparent Lending Decisions.

#### Purpose:
A time-based feature store integration enables machine learning pipelines to perform point-in-time correct data retrieval—known as "time travel"—by synchronizing historical offline records with fresh, low-latency online feature values

In [208]:
# STANDARD LIBRARY
# ------------------------------------

from __future__ import annotations

import json
import logging
import warnings
import hashlib
import platform

from pathlib import Path
from datetime import datetime
from dataclasses import dataclass

# DATA PROCESSING
# --------------------------------------

import numpy as np
import pandas as pd

# SCIPY
# --------------------------------------

from scipy import sparse
from scipy.sparse import load_npz

# VISUALISATION
# --------------------------------------

import matplotlib.pyplot as plt
import seaborn as sns

# REPRODUCIBILITY
# --------------------------------------

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore")

# DISPLAY CONFIGURATION
# ------------------------------------

pd.set_option("display.max_columns", None)

pd.set_option("display.width", 180)

pd.set_option("display.max_colwidth", 120)

In [209]:
# NOTEBOOK INFORMATION
# -------------------------------------------------------------

PROJECT_NAME = "Explainable AI Credit Risk Decision Platform"

NOTEBOOK_NAME = "09_time_based_feature_store_integration"

PIPELINE_STAGE = "Matrix D"

PIPELINE_VERSION = "1.0.0"

AUTHOR = "Emmanuel Ahadzi"

CREATED = datetime.now().strftime("%Y-%m-%d")

In [211]:
# INPUT FILES
# ----------------------------------------------------------------------

MATRIX_C_TRAIN = FEATURE_STORE_DIR / "matrix_c_train.npz"

MATRIX_C_TEST = FEATURE_STORE_DIR / "matrix_c_test.npz"

FEATURE_NAMES = FEATURE_STORE_DIR / "feature_names.csv"

MACRO_FEATURE_STORE = FEATURE_STORE_DIR / "macroeconomic_feature_store.parquet"

PROCESSED_LOANS = PROCESSED_DIR / "processed_loans.parquet"

In [212]:
# OUTPUT FILES
# ----------------------------------------------

MATRIX_D_DIR = FEATURE_STORE_DIR / "matrix_d"

MATRIX_D_DIR.mkdir(parents=True, exist_ok=True)

In [213]:
# ENVIRONMENT INFORMATION
# ----------------------------------------

ENVIRONMENT = {

    "Python": platform.python_version(),

    "Operating System": platform.system(),

    "Machine": platform.machine(),

    "Processor": platform.processor()}

In [214]:
# LOGGING CONFIGURATION
# -----------------------------------------------

LOG_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / f"{NOTEBOOK_NAME}.log"

logging.basicConfig(level=logging.INFO,
                    
    format="%(asctime)s | %(levelname)s | %(message)s",
                    
    handlers=[logging.FileHandler(LOG_FILE)
              
              ,logging.StreamHandler()],force=True)

logger = logging.getLogger(__name__)

logger.info("Notebook Started")

2026-07-16 21:53:53,955 | INFO | Notebook Started


In [215]:
# DISPLAY SECTION TITLE
# ---------------------------------

def section(title: str):
    """Display a formatted notebook section header."""

    print("\n")
    
    print(title.upper())

    logger.info(title)

In [216]:
# UNIVERSAL DATASET LOADER
# ----------------------------

def load_dataset(path: Path):
    
    logger.info(f"Loading {path.name}")

    if not path.exists():
        raise FileNotFoundError(f"{path} not found.")

    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)

    elif suffix == ".parquet":
        return pd.read_parquet(path)

    elif suffix == ".npz":
        return load_npz(path)

    else:
        raise ValueError(f"Unsupported file type: {suffix}")

In [217]:
# DATASET SUMMARY
# ------------------------------------

def dataset_summary(df: pd.DataFrame, name: str):

    summary = {

        "Dataset": name,

        "Rows": df.shape[0],

        "Columns": df.shape[1],

        "Missing Values": int(df.isna().sum().sum()),

        "Duplicate Rows": int(df.duplicated().sum())}

    display(pd.DataFrame(summary, index=[0]))

    logger.info(summary)

    return summary

In [218]:
# DATA VALIDATION
# --------------------------------------------

def validate_dataframe(df: pd.DataFrame,
                       name: str,
                       key: str = None):
    report = {

        "Dataset": name,

        "Rows": df.shape[0],

        "Columns": df.shape[1],

        "Missing Values": int(df.isna().sum().sum()),

        "Duplicate Rows": int(df.duplicated().sum())}

    if key is not None:

        report["Missing Key"] = int(df[key].isna().sum())

        report["Unique Keys"] = df[key].nunique()

    report = pd.DataFrame(report, index=[0])

    return report

In [219]:
# EXPORT REPORT
# --------------------------------------------

def save_report(df: pd.DataFrame,
                
                filename: str,
                
                folder: Path = REPORT_DIR):
    """
    Export DataFrame as CSV.
    """

    folder.mkdir(parents=True, exist_ok=True)

    output = folder / filename

    df.to_csv(output, index=False)

    logger.info(f"Saved: {output.name}")

In [220]:
# DATASET CHECKSUM

def dataset_checksum(df: pd.DataFrame):

    checksum = hashlib.md5(

        pd.util.hash_pandas_object(

            df,index=True).values).hexdigest()

    return checksum

In [221]:
# NOTEBOOK COMPLETION
# -------------------------------

def notebook_complete():

    logger.info("Notebook Completed Successfully")

    logger.info("=" * 60)

    print("NOTEBOOK COMPLETED SUCCESSFULLY")

In [222]:
# Load Processed Lending Club Dataset

CLEAN_LOAN_PATH = PROCESSED_DIR / "clean_processed_lending_club.parquet"

loan_df = pd.read_parquet(CLEAN_LOAN_PATH)

print(loan_df.shape)
display(loan_df.head())

(500000, 12)


,id,loan_status,issue_d,loan_amnt,term,grade,int_rate,annual_inc,dti,home_ownership,title,revol_util
0,68407277,Fully Paid,2015-12-01,3600.0,36 months,C,13.99,55000.0,5.91,MORTGAGE,Debt consolidation,29.7
1,68355089,Fully Paid,2015-12-01,24700.0,36 months,C,11.99,65000.0,16.06,MORTGAGE,Business,19.2
2,68341763,Fully Paid,2015-12-01,20000.0,60 months,B,10.78,63000.0,10.78,MORTGAGE,Debt consolidation,56.2
3,66310712,Current,2015-12-01,35000.0,60 months,C,14.85,110000.0,17.06,MORTGAGE,Debt consolidation,11.6
4,68476807,Fully Paid,2015-12-01,10400.0,60 months,F,22.45,104433.0,25.37,MORTGAGE,Major purchase,64.5


In [223]:
# DATASET OVERVIEW

loan_summary = dataset_summary(loan_df,"clean_processed_lending_club.parquet")

,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,clean_processed_lending_club.parquet,500000,12,0,0


2026-07-16 21:54:18,844 | INFO | {'Dataset': 'clean_processed_lending_club.parquet', 'Rows': 500000, 'Columns': 12, 'Missing Values': 0, 'Duplicate Rows': 0}


In [224]:
# REQUIRED COLUMN VALIDATION
# -----------------------------------------

required_columns = ["loan_status",
                    
                    "issue_d",
                    
                    "title"]

missing_columns = [column
                   
                   for column in required_columns
                   
                   if column not in loan_df.columns]

if missing_columns:

    raise ValueError(f"Missing required columns: {missing_columns}")

logger.info("Required columns successfully validated.")

2026-07-16 21:54:21,169 | INFO | Required columns successfully validated.


In [225]:
# DATASET INFORMATION
# --------------------------------------

print("Processed Lending Club Dataset")

print(f"Rows       : {loan_df.shape[0]:,}")

print(f"Columns    : {loan_df.shape[1]:,}")

print(f"Memory (MB): {loan_df.memory_usage(deep=True).sum()/1024**2:.2f}")

Processed Lending Club Dataset
Rows       : 500,000
Columns    : 12
Memory (MB): 84.04


In [226]:
# TARGET DISTRIBUTION
# -----------------------------

target_distribution = (loan_df["loan_status"]

    .value_counts(normalize=True)

    .mul(100)

    .round(2)

    .rename("Percentage")

    .reset_index())

target_distribution.columns = ["Loan Status",
                               
                               "Percentage"]

display(target_distribution)

,Loan Status,Percentage
0,Fully Paid,62.47
1,Current,20.85
2,Charged Off,15.76
3,Late (31-120 days),0.60
4,In Grace Period,0.21
5,Late (16-30 days),0.11
6,Default,0.00


In [227]:
# MISSING VALUE SUMMARY
# --------------------------------

missing_summary = (loan_df

    .isna()

    .sum()

    .sort_values(ascending=False)

    .rename("Missing Values")

    .reset_index())

missing_summary.columns = ["Feature","Missing Values"]

display(missing_summary.head(15))

,Feature,Missing Values
0,id,0
1,loan_status,0
2,issue_d,0
3,loan_amnt,0
4,term,0
5,grade,0
6,int_rate,0
7,annual_inc,0
8,dti,0
9,home_ownership,0


In [228]:
# EXPORT REPORTS
# -----------------------------

save_report(loan_validation,"loan_validation_report.csv")

save_report(target_distribution,"loan_target_distribution.csv")

save_report(missing_summary,"loan_missing_summary.csv")

# SECTION COMPLETE
# ------------------------------------------------

logger.info("SECTION 3 COMPLETED")

print("\n✓ Processed Lending Club Dataset successfully loaded and validated.")

2026-07-16 21:54:29,203 | INFO | Saved: loan_validation_report.csv
2026-07-16 21:54:29,207 | INFO | Saved: loan_target_distribution.csv
2026-07-16 21:54:29,211 | INFO | Saved: loan_missing_summary.csv
2026-07-16 21:54:29,216 | INFO | SECTION 3 COMPLETED



✓ Processed Lending Club Dataset successfully loaded and validated.


#### Load Production Macroeconomic Feature Store
##### Purpose
##### -----------------------------------------------------
##### Load the validated Production Macroeconomic Feature Store created in
##### Notebook 08.

In [229]:
# LOCATE FEATURE STORE
# -----------------------------------

MACRO_DATA_PATH = FEATURE_STORE_DIR / "macroeconomic_feature_store.parquet"

logger.info(f"Feature Store Path: {MACRO_DATA_PATH}")

2026-07-16 21:54:32,697 | INFO | Feature Store Path: /Users/emmanuelahadzi/data/feature_store/macroeconomic_feature_store.parquet


In [230]:
# LOAD FEATURE STORE
# ------------------------------------

macro_df = load_dataset(MACRO_DATA_PATH)

logger.info("Production Macroeconomic Feature Store loaded successfully.")

2026-07-16 21:54:34,541 | INFO | Loading macroeconomic_feature_store.parquet
2026-07-16 21:54:34,557 | INFO | Production Macroeconomic Feature Store loaded successfully.


In [231]:
# FEATURE STORE SUMMARY
# ---------------------------------------

macro_summary = dataset_summary(macro_df,"Macroeconomic Feature Store")

# DATA PREVIEW
# --------------------------------------

display(macro_df.head())

,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,Macroeconomic Feature Store,144,25,0,0


2026-07-16 21:54:36,925 | INFO | {'Dataset': 'Macroeconomic Feature Store', 'Rows': 144, 'Columns': 25, 'Missing Values': 0, 'Duplicate Rows': 0}


,DATE,Year,Month,YearMonth,FEDFUNDS,UNRATE,DGS10,UMCSENT,FEDFUNDS_MoM,CPIAUCSL_YoY,CPIAUCSL_3M_Momentum,GDPC1_Growth,YIELD_CURVE,Articles,LOAN,MORTGAGE,FEDERAL_RESERVE,ECONOMY,Average_Tone,Average_Tone_LAG3,UNRATE_CHANGE,Average_Tone_CHANGE,ECONOMIC_STRESS_INDEX,RECESSION_FLAG,HIGH_RATE_ENVIRONMENT
0,2007-01-01,2007,1,2007-01,5.25,4.6,4.759524,96.9,0.01,4.294696,1.212660,0.000000,-0.116667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.1,0.0,14.144696,0,1
1,2007-02-01,2007,2,2007-02,5.26,4.5,4.722632,91.3,0.01,4.294696,1.212660,0.000000,-0.126316,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.1,0.0,14.054696,0,1
2,2007-03-01,2007,3,2007-03,5.26,4.4,4.564545,88.4,0.00,4.294696,1.212660,0.000000,-0.009091,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.1,0.0,13.954696,0,1
3,2007-04-01,2007,4,2007-04,5.25,4.5,4.693810,87.1,-0.01,4.294696,1.212660,0.611762,0.027619,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.0,14.044696,0,1
4,2007-05-01,2007,5,2007-05,5.25,4.4,4.746364,88.3,0.00,4.294696,1.238334,0.000000,-0.020000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.1,0.0,13.944696,0,1


In [232]:
# DATA VALIDATION
# ---------------------------------------

macro_validation = validate_dataframe(df=macro_df,
                                      
                                      name="Macroeconomic Feature Store",
                                      
                                      key="YearMonth")

display(macro_validation)

,Dataset,Rows,Columns,Missing Values,Duplicate Rows,Missing Key,Unique Keys
0,Macroeconomic Feature Store,144,25,0,0,0,144


In [233]:
# REQUIRED COLUMN VALIDATION
# ---------------------------------

required_columns = ["YearMonth"]

missing_columns = [

    col

    for col in required_columns

    if col not in macro_df.columns]

if missing_columns:

    raise ValueError(f"Missing required columns: {missing_columns}")

logger.info("Required key columns successfully validated.")

2026-07-16 21:54:39,358 | INFO | Required key columns successfully validated.


In [234]:
# DATE COVERAGE
# --------------------------------------

date_coverage = pd.DataFrame({

    "Metric": [

        "Start Period",

        "End Period",

        "Unique Months"],

    "Value": [

        macro_df["YearMonth"].min(),

        macro_df["YearMonth"].max(),

        macro_df["YearMonth"].nunique()]})

display(date_coverage)

,Metric,Value
0,Start Period,2007-01
1,End Period,2018-12
2,Unique Months,144


In [235]:
# MISSING VALUE PROFILE
# -------------------------------

macro_missing = (

    macro_df

    .isna()

    .sum()

    .sort_values(ascending=False)

    .rename("Missing Values")

    .reset_index())

macro_missing.columns = ["Feature",
                         
                         "Missing Values"]

display(macro_missing.head(20))

,Feature,Missing Values
0,DATE,0
1,Articles,0
2,RECESSION_FLAG,0
3,ECONOMIC_STRESS_INDEX,0
4,Average_Tone_CHANGE,0
5,UNRATE_CHANGE,0
6,Average_Tone_LAG3,0
7,Average_Tone,0
8,ECONOMY,0
9,FEDERAL_RESERVE,0


In [236]:
# DATA TYPES
# ----------------------------------

dtype_report = pd.DataFrame({

    "Feature": macro_df.columns,

    "Data Type": macro_df.dtypes.astype(str)})

display(dtype_report.head(15))

,Feature,Data Type
DATE,DATE,datetime64[ns]
Year,Year,int64
Month,Month,int64
YearMonth,YearMonth,object
FEDFUNDS,FEDFUNDS,float64
UNRATE,UNRATE,float64
DGS10,DGS10,float64
UMCSENT,UMCSENT,float64
FEDFUNDS_MoM,FEDFUNDS_MoM,float64
CPIAUCSL_YoY,CPIAUCSL_YoY,float64


In [237]:
# MEMORY REPORT
# -----------------------------

memory_mb = macro_df.memory_usage(deep=True).sum() / 1024**2


print("Feature Store Information")
print("="*40)
print(f"Rows        : {macro_df.shape[0]:,}")
print(f"Columns     : {macro_df.shape[1]:,}")
print(f"Memory (MB) : {memory_mb:.2f}")


Feature Store Information
Rows        : 144
Columns     : 25
Memory (MB) : 0.03


In [238]:
# EXPORT REPORTS
# --------------------------------

save_report(macro_validation,"macro_validation_report.csv")

save_report(macro_missing,"macro_missing_summary.csv")

save_report(date_coverage,"macro_date_coverage.csv")

save_report(dtype_report,"macro_dtype_report.csv")

# SECTION COMPLETE
# ------------------------------------

logger.info("SECTION 4 COMPLETED")

print("\n✓ Production Macroeconomic Feature Store successfully loaded and validated.")

2026-07-16 21:54:46,880 | INFO | Saved: macro_validation_report.csv
2026-07-16 21:54:46,883 | INFO | Saved: macro_missing_summary.csv
2026-07-16 21:54:46,886 | INFO | Saved: macro_date_coverage.csv
2026-07-16 21:54:46,889 | INFO | Saved: macro_dtype_report.csv
2026-07-16 21:54:46,889 | INFO | SECTION 4 COMPLETED



✓ Production Macroeconomic Feature Store successfully loaded and validated.


#### Time Dimension Engineering

##### Purpose
###### ------------------------------------------------------------------------
###### Create a standardised temporal key (YearMonth) from the loan issue date
###### to enable accurate integration with the monthly Macroeconomic Feature Store.

In [239]:
# CONVERT ISSUE DATE
# -----------------------------------------------

logger.info("Converting issue_d to datetime...")

loan_df["issue_d"] = pd.to_datetime(loan_df["issue_d"],

    errors="coerce")

print(f"Missing Dates : {loan_df['issue_d'].isna().sum():,}")

# DATE VALIDATION
# -----------------------------------------------

if loan_df["issue_d"].isna().sum() > 0:

    logger.warning("Some issue dates could not be converted.")

else:

    logger.info("All issue dates converted successfully.")

2026-07-16 21:54:50,863 | INFO | Converting issue_d to datetime...
2026-07-16 21:54:50,952 | INFO | All issue dates converted successfully.


Missing Dates : 0


In [240]:
# CALENDAR DIMENSION
# ------------------------------------------------------

loan_df["Year"] = loan_df["issue_d"].dt.year

loan_df["Month"] = loan_df["issue_d"].dt.month

loan_df["Quarter"] = loan_df["issue_d"].dt.quarter

loan_df["Month_Name"] = loan_df["issue_d"].dt.month_name()

loan_df["YearMonth"] = loan_df["issue_d"].dt.to_period("M").astype(str)

logger.info("Calendar features created.")

2026-07-16 21:54:54,662 | INFO | Calendar features created.


In [241]:
# CALENDAR PREVIEW
# --------------------------------------

display(loan_df[[

            "issue_d",

            "Year",

            "Month",

            "Quarter",

            "Month_Name",

            "YearMonth"]].head())

,issue_d,Year,Month,Quarter,Month_Name,YearMonth
0,2015-12-01,2015,12,4,December,2015-12
1,2015-12-01,2015,12,4,December,2015-12
2,2015-12-01,2015,12,4,December,2015-12
3,2015-12-01,2015,12,4,December,2015-12
4,2015-12-01,2015,12,4,December,2015-12


In [242]:
# JOIN KEY VALIDATION
# -------------------------------------------------------------

missing_keys = loan_df["YearMonth"].isna().sum()

duplicate_keys = loan_df["YearMonth"].duplicated().sum()

print("="*40)

print(f"Missing Join Keys   : {missing_keys:,}")

print(f"Duplicate Join Keys : {duplicate_keys:,}")

print("="*40)

Missing Join Keys   : 0
Duplicate Join Keys : 499,985


In [243]:
# DATE COVERAGE COMPARISON
# ------------------------------------------------------

loan_months = set(loan_df["YearMonth"].unique())

macro_months = set(macro_df["YearMonth"].unique())

matched_months = loan_months.intersection(macro_months)

missing_macro_months = loan_months.difference(macro_months)

coverage_report = pd.DataFrame({

    "Metric": [

        "Loan Months",

        "Macro Months",

        "Matched Months",

        "Loan Months Without Macro Data"],

    "Value": [

        len(loan_months),

        len(macro_months),

        len(matched_months),

        len(missing_macro_months)]})

display(coverage_report)

,Metric,Value
0,Loan Months,15
1,Macro Months,144
2,Matched Months,15
3,Loan Months Without Macro Data,0


In [244]:
# MISSING MACRO MONTHS
# -------------------------------------------------------

missing_months = pd.DataFrame({"YearMonth": sorted(missing_macro_months)})

display(missing_months.head())

,YearMonth


In [245]:
# CALENDAR DIMENSION REPORT
# -----------------------------------

calendar_dimension = (loan_df
    
    .groupby(

        ["Year", "Quarter", "Month", "Month_Name", "YearMonth"])

    .size()

    .reset_index(name="Loan_Count"))

display(calendar_dimension.head())

,Year,Quarter,Month,Month_Name,YearMonth,Loan_Count
0,2015,1,1,January,2015-01,35107
1,2015,1,2,February,2015-02,23770
2,2015,1,3,March,2015-03,25400
3,2015,2,4,April,2015-04,35427
4,2015,2,5,May,2015-05,31913


In [246]:
# EXPORT REPORTS
# ------------------------------------

save_report(calendar_dimension,"loan_calendar_dimension.csv")

save_report(coverage_report,"date_validation_report.csv")

save_report(missing_months,"missing_macro_months.csv")

2026-07-16 21:55:07,196 | INFO | Saved: loan_calendar_dimension.csv
2026-07-16 21:55:07,200 | INFO | Saved: date_validation_report.csv
2026-07-16 21:55:07,202 | INFO | Saved: missing_macro_months.csv


In [247]:
# SECTION SUMMARY
# ___________________________

print("TIME-BASED JOIN KEY CREATED")

print("="*50)

print(f"Loan Records          : {loan_df.shape[0]:,}")

print(f"Unique Loan Months    : {len(loan_months):,}")

print(f"Macro Months          : {len(macro_months):,}")

print(f"Matched Months        : {len(matched_months):,}")

print(f"Missing Macro Months  : {len(missing_macro_months):,}")

logger.info("Section 5 completed successfully.")

2026-07-16 21:55:08,174 | INFO | Section 5 completed successfully.


TIME-BASED JOIN KEY CREATED
Loan Records          : 500,000
Unique Loan Months    : 15
Macro Months          : 144
Matched Months        : 15
Missing Macro Months  : 0


In [260]:
# OPTIMISE NUMERIC DATA TYPES
# _______________________________________________

numeric_columns = matrix_d.select_dtypes(

    include=["int64", "float64"]).columns

matrix_d[numeric_columns] = matrix_d[numeric_columns].apply(

    pd.to_numeric,downcast="float")

logger.info("Numeric optimisation completed.")

2026-07-16 21:56:21,439 | INFO | Numeric optimisation completed.


In [261]:
# OPTIMISE CATEGORICAL DATA TYPES
# ________________________________________________

categorical_columns = matrix_d.select_dtypes(

    include="object").columns

for col in categorical_columns:

    if matrix_d[col].nunique() < 100:

        matrix_d[col] = matrix_d[col].astype("category")

logger.info("Categorical optimisation completed.")

2026-07-16 21:56:22,953 | INFO | Categorical optimisation completed.


In [262]:
# REMOVE CONSTANT FEATURES
# ________________________________________________________________

constant_features = [col
                     
                     for col in matrix_d.columns
                         
                     if matrix_d[col].nunique(dropna=False) <= 1]

matrix_d.drop(columns=constant_features,inplace=True)

logger.info(f"Removed {len(constant_features)} constant features.")

2026-07-16 21:56:25,662 | INFO | Removed 1 constant features.


In [263]:
# MISSING VALUE REPORT
# _________________________________________

missing_report = (matrix_d

    .isna()

    .sum()

    .reset_index())

missing_report.columns = [

    "Feature",

    "Missing Values"]

missing_report["Missing %"] = (

    missing_report["Missing Values"]

    / len(matrix_d)* 100).round(2)

display(missing_report.head(20))

,Feature,Missing Values,Missing %
0,id,0,0.0
1,loan_status,0,0.0
2,issue_d,0,0.0
3,loan_amnt,0,0.0
4,term,0,0.0
5,grade,0,0.0
6,int_rate,0,0.0
7,annual_inc,0,0.0
8,dti,0,0.0
9,home_ownership,0,0.0


In [264]:
# MEMORY REPORT

memory_before = matrix_d.memory_usage(deep=True).sum() / 1024**2

memory_report = pd.DataFrame({

    "Rows": [matrix_d.shape[0]],

    "Columns": [matrix_d.shape[1]],

    "Memory (MB)": [round(memory_before, 2)]})

display(memory_report)

,Rows,Columns,Memory (MB)
0,500000,38,94.97


In [265]:
# FEATURE INVENTORY
# _________________________________________

feature_inventory = pd.DataFrame({

    "Feature": matrix_d.columns,

    "Data Type": matrix_d.dtypes.astype(str)})

display(feature_inventory.head())

,Feature,Data Type
id,id,string
loan_status,loan_status,category
issue_d,issue_d,datetime64[ns]
loan_amnt,loan_amnt,float32
term,term,category


In [266]:
# EXPORT REPORTS
# _______________________________________________

save_report(feature_inventory,"matrix_d_feature_inventory.csv")

save_report(missing_report,"matrix_d_missing_report.csv")

save_report(memory_report,"matrix_d_memory_report.csv")

2026-07-16 21:56:46,264 | INFO | Saved: matrix_d_feature_inventory.csv
2026-07-16 21:56:46,268 | INFO | Saved: matrix_d_missing_report.csv
2026-07-16 21:56:46,271 | INFO | Saved: matrix_d_memory_report.csv


In [267]:
# FINAL VALIDATION
# _____________________________________

print("PRODUCTION FEATURE PREPARATION COMPLETE")

print("=" * 90)

print(f"Rows                 : {matrix_d.shape[0]:,}")

print(f"Columns              : {matrix_d.shape[1]:,}")

print(f"Constant Features    : {len(constant_features)}")

print(f"Numeric Features     : {len(matrix_d.select_dtypes(include='number').columns)}")

print(f"Categorical Features : {len(matrix_d.select_dtypes(include='category').columns)}")

PRODUCTION FEATURE PREPARATION COMPLETE
Rows                 : 500,000
Columns              : 38
Constant Features    : 1
Numeric Features     : 29
Categorical Features : 6


In [295]:
# SECTION SUMMARY
# ________________________________________________

print("="*90)

print("TIME-BASED SPLIT COMPLETED")

print("="*90)

print(f"Training Rows : {len(train_df):,}")

print(f"Testing Rows  : {len(test_df):,}")

print(f"Training End  : {train_df['issue_d'].max()}")

print(f"Testing Start : {test_df['issue_d'].min()}")

print("="*90)

logger.info("Section 8 completed successfully.")

2026-07-16 22:35:07,682 | INFO | Section 8 completed successfully.


TIME-BASED SPLIT COMPLETED
Training Rows : 421,097
Testing Rows  : 78,903
Training End  : 2015-12-01 00:00:00
Testing Start : 2018-01-01 00:00:00


In [188]:
# FINAL STRUCTURED FEATURES
# ________________________________

structured_features = (

    borrower_features +

    loan_features +

    fred_features +

    gdelt_features)

target = "loan_status"

In [189]:
print(f"Total Structured Features : {len(structured_features)}")

display(pd.DataFrame({"Structured Features": structured_features}))

Total Structured Features : 17


,Structured Features
0,annual_inc
1,dti
2,home_ownership
3,revol_util
4,loan_amnt
5,term
6,grade
7,int_rate
8,FEDFUNDS
9,UNRATE
